# **Coleta de Dados do StatsBomb**

## **1. Configuracao do Ambiente**

In [1]:
import os
import time
import warnings
from pathlib import Path
from typing import List, Tuple, Optional, Union

import numpy as np
import pandas as pd
from statsbombpy import sb

warnings.filterwarnings('ignore')

## **2. Configuracao**

In [2]:
DATA_PATH = Path("../data")
RAW_DATA_PATH = DATA_PATH / "raw"
COMBINED_DATA_PATH = RAW_DATA_PATH / "combined"
VIZ_PATH = Path("../viz")
MATCHES_PATH = RAW_DATA_PATH / "matches"
EVENTS_PATH = RAW_DATA_PATH / "events"
for path in [MATCHES_PATH, EVENTS_PATH, COMBINED_DATA_PATH, VIZ_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Diretorios de dados criados:")
print(f"  Matches: {MATCHES_PATH}")
print(f"  Events: {EVENTS_PATH}")
print(f"  Combined: {COMBINED_DATA_PATH}")
print(f"  Viz: {VIZ_PATH}")

Diretorios de dados criados:
  Matches: ../data/raw/matches
  Events: ../data/raw/events
  Combined: ../data/raw/combined
  Viz: ../viz


## **3. Selecao de Competicao e Temporada**

In [3]:
COMPETITIONS_SEASONS = [
    (9, 27),
    (11, 27),
    (2, 27),
    (12, 27),
    (7, 27)
]

print(f"Total de competicoes/temporadas para coletar: {len(COMPETITIONS_SEASONS)}")
print(f"Pares selecionados: {COMPETITIONS_SEASONS}")

Total de competicoes/temporadas para coletar: 5
Pares selecionados: [(9, 27), (11, 27), (2, 27), (12, 27), (7, 27)]


## **4. Funcoes Utilitarias**

In [4]:
def generate_filename(competition_id: int, season_id: int, data_type: str = "matches") -> str:
    return f"{data_type}_competition_{competition_id}_season_{season_id}.parquet"


def check_existing_files(path: Path, prefix: str) -> List[Tuple[int, int]]:
    existing_pairs = []
    
    for filename in os.listdir(path):
        if filename.endswith('.parquet') and filename.startswith(prefix):
            parts = filename.replace('.parquet', '').split('_')
            if len(parts) >= 5:
                try:
                    competition_id = int(parts[2])
                    season_id = int(parts[4])
                    existing_pairs.append((competition_id, season_id))
                except (ValueError, IndexError):
                    print(f"Aviso: formato de nome de arquivo invalido: {filename}")
    
    return existing_pairs


def fetch_with_retry(fetch_function, *args, max_retries: int = 3, **kwargs) -> Optional[pd.DataFrame]:
    for attempt in range(max_retries):
        try:
            data = fetch_function(*args, **kwargs)
            return data
        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = (attempt + 1) * 2
                print(f"  Tentativa {attempt + 1} falhou: {e}")
                print(f"  Tentando novamente em {wait_time} segundos...")
                time.sleep(wait_time)
            else:
                print(f"  Falhou apos {max_retries} tentativas: {e}")
                return None
    
    return None


MATCH_ID_COLUMNS = [
    'home_manager_id',
    'home_manager_country_id',
    'away_manager_id',
    'away_manager_country_id',
]


def normalize_match_id_columns(matches_df: pd.DataFrame) -> pd.DataFrame:
    normalized = matches_df.copy()

    for column in MATCH_ID_COLUMNS:
        if column not in normalized.columns:
            continue

        def normalize_id(value):
            if pd.isna(value):
                return pd.NA
            if isinstance(value, (float, np.floating)) and value.is_integer():
                return str(int(value))
            return str(value)

        normalized[column] = normalized[column].map(normalize_id).astype('string')

    return normalized


def ensure_coordinate_columns(events_df: pd.DataFrame) -> pd.DataFrame:
    coord_specs = [
        ('location', ('location_x', 'location_y')),
        ('pass_end_location', ('pass_end_location_x', 'pass_end_location_y')),
        ('carry_end_location', ('carry_end_location_x', 'carry_end_location_y')),
    ]
    for src_col, (x_col, y_col) in coord_specs:
        if src_col not in events_df.columns:
            continue
        needs_backfill = (
            x_col not in events_df.columns or
            y_col not in events_df.columns or
            events_df[[x_col, y_col]].isna().all().all()
        )
        if needs_backfill:
            events_df[[x_col, y_col]] = events_df[src_col].apply(
                lambda x: pd.Series(x[:2]) if isinstance(x, (tuple, list, np.ndarray)) else pd.Series([np.nan, np.nan])
            )
    return events_df

## **5. Coleta de Dados de Partidas**

In [5]:
def collect_match_data() -> pd.DataFrame:
    existing_matches = check_existing_files(MATCHES_PATH, "matches_competition")
    print(f"Encontrados {len(existing_matches)} arquivos de partidas ja existentes.")
    
    all_matches_dfs = []
    
    for competition_id, season_id in COMPETITIONS_SEASONS:
        if (competition_id, season_id) in existing_matches:
            print(f"\nCarregando dados existentes: Competicao {competition_id}, Temporada {season_id}")
            parquet_path = MATCHES_PATH / generate_filename(competition_id, season_id)
            matches_df = pd.read_parquet(parquet_path)
            matches_df = normalize_match_id_columns(matches_df)
            all_matches_dfs.append(matches_df)
        else:
            print(f"\nBuscando novos dados: Competicao {competition_id}, Temporada {season_id}")
            matches_df = fetch_with_retry(
                sb.matches,
                competition_id=competition_id,
                season_id=season_id
            )
            
            if matches_df is not None:
                matches_df = normalize_match_id_columns(matches_df)
                parquet_path = MATCHES_PATH / generate_filename(competition_id, season_id)
                matches_df.to_parquet(parquet_path, index=False)
                print(f"  Salvo em: {parquet_path.name}")
                all_matches_dfs.append(matches_df)
            
            time.sleep(1) 
    
    if all_matches_dfs:
        all_matches = pd.concat(all_matches_dfs, ignore_index=True)
        all_matches = normalize_match_id_columns(all_matches)
        print(f"\nTotal de partidas coletadas: {len(all_matches)}")
        
        combined_path = COMBINED_DATA_PATH / "all_matches_combined.parquet"
        all_matches.to_parquet(combined_path, index=False)
        print(f"Arquivo consolidado salvo em: {combined_path}")
        
        return all_matches
    else:
        print("\nNenhum dado de partida coletado")
        return pd.DataFrame()

In [6]:
all_matches = collect_match_data()

Encontrados 40 arquivos de partidas ja existentes.

Carregando dados existentes: Competicao 9, Temporada 27

Carregando dados existentes: Competicao 11, Temporada 27

Carregando dados existentes: Competicao 2, Temporada 27

Carregando dados existentes: Competicao 12, Temporada 27

Carregando dados existentes: Competicao 7, Temporada 27

Total de partidas coletadas: 1551
Arquivo consolidado salvo em: ../data/raw/combined/all_matches_combined.parquet


## **6. Coleta de Dados de Eventos**

In [7]:
def collect_event_data(match_ids: List[int]) -> pd.DataFrame:
    existing_events = []
    for filename in os.listdir(EVENTS_PATH):
        if filename.endswith('.parquet') and filename.startswith('events_match_'):
            try:
                match_id = int(filename.replace('events_match_', '').replace('.parquet', ''))
                existing_events.append(match_id)
            except ValueError:
                pass
    
    print(f"Encontrados {len(existing_events)} arquivos de eventos ja existentes.")
    
    all_events_dfs = []
    total_matches = len(match_ids)
    
    for idx, match_id in enumerate(match_ids):
        if match_id in existing_events:
            events_path = EVENTS_PATH / f"events_match_{match_id}.parquet"
            events_df = pd.read_parquet(events_path)
            all_events_dfs.append(events_df)
        else:
            print(f"\nBuscando eventos da partida {match_id}")
            events_df = fetch_with_retry(sb.events, match_id=match_id)
            
            if events_df is not None:
                events_path = EVENTS_PATH / f"events_match_{match_id}.parquet"
                events_df.to_parquet(events_path, index=False)
                all_events_dfs.append(events_df)
            
            time.sleep(1) 
        
        if (idx + 1) % 50 == 0:
            print(f"\nProgresso: {idx + 1}/{total_matches} partidas processadas")
    
    if all_events_dfs:
        print("\nConsolidando todos os eventos...")
        all_events = pd.concat(all_events_dfs, ignore_index=True)
        all_events = ensure_coordinate_columns(all_events)
        all_events = all_events.sort_values(['match_id', 'period', 'minute', 'second', 'index']).reset_index(drop=True)
        print(f"Total de eventos brutos coletados: {len(all_events)}")

        combined_path = COMBINED_DATA_PATH / "all_events_combined.parquet"
        all_events.to_parquet(combined_path, index=False)
        print(f"Arquivo consolidado de eventos brutos salvo em: {combined_path}")
        
        return all_events
    else:
        print("\nNenhum dado de evento coletado")
        return pd.DataFrame()

In [8]:
if not all_matches.empty:
    match_ids = all_matches['match_id'].tolist()
    print(f"\nTotal de partidas: {len(match_ids)}")
    
    all_events = collect_event_data(match_ids)
else:
    print("Nenhuma partida disponivel para a coleta de eventos")
    all_events = pd.DataFrame()


Total de partidas: 1551
Encontrados 2651 arquivos de eventos ja existentes.

Progresso: 50/1551 partidas processadas

Progresso: 100/1551 partidas processadas

Progresso: 150/1551 partidas processadas

Progresso: 200/1551 partidas processadas

Progresso: 250/1551 partidas processadas

Progresso: 300/1551 partidas processadas

Progresso: 350/1551 partidas processadas

Progresso: 400/1551 partidas processadas

Progresso: 450/1551 partidas processadas

Progresso: 500/1551 partidas processadas

Progresso: 550/1551 partidas processadas

Progresso: 600/1551 partidas processadas

Progresso: 650/1551 partidas processadas

Progresso: 700/1551 partidas processadas

Progresso: 750/1551 partidas processadas

Progresso: 800/1551 partidas processadas

Progresso: 850/1551 partidas processadas

Progresso: 900/1551 partidas processadas

Progresso: 950/1551 partidas processadas

Progresso: 1000/1551 partidas processadas

Progresso: 1050/1551 partidas processadas

Progresso: 1100/1551 partidas processad